In [3]:
import numpy as np
from scipy.stats import unitary_group

def digitize_real_number(x, L):
    """Convert a real number x in [0,1) to its binary representation as a list of bits."""
    binary_repr = f"{int(x * (2**L)):0{L}b}"
    return np.array([int(bit) for bit in binary_repr], dtype=int)

def apply_cyclic_leftward_shift_to_state(state, L):
    """
    Applies the cyclic leftward shift to each computational basis state.
    
    Parameters:
    state (np.ndarray): The state vector in the computational basis (2^L dimensional).
    L (int): The number of qubits.
    
    Returns:
    np.ndarray: The new state vector after applying the cyclic leftward shift.
    """
    # Initialize the new state as a zero vector
    new_state = np.zeros_like(state, dtype=complex)
    
    # Iterate over each computational basis state
    for i in range(2**L):
        # Convert index to binary representation (bit string)
        bit_string = f"{i:0{L}b}"
        
        # Apply the cyclic leftward shift to the bit string
        shifted_bit_string = bit_string[1:] + bit_string[0]
        
        # Convert the shifted bit string back to an integer index
        shifted_index = int(shifted_bit_string, 2)
        
        # Update the new state vector
        new_state[shifted_index] = state[i]
    
    return new_state

def get_last_two_qubit_indices(time, L):
    """
    Returns the indices of the last two qubits after a specified number of cyclic leftward shifts.
    
    Parameters:
    time (int): The current time step (number of cyclic leftward shifts applied).
    L (int): The total number of qubits.
    
    Returns:
    tuple: A tuple containing the indices of the last two qubits.
    """
    # Calculate the indices of the last two qubits after the shifts
    qubit1 = (L - 2 + time) % L
    qubit2 = (L - 1 + time) % L
    
    return qubit1, qubit2 
 
def embed_haar_to_full_unitary(L, target_qubits):
    """
    Embeds a Haar random matrix into a full unitary matrix for a system of L qubits.
    
    Parameters:
    L (int): Total number of qubits in the full system.
    target_qubits (list): List of two qubits where the Haar random unitary should be applied.
    
    Returns:
    np.ndarray: A full (2^L x 2^L) unitary matrix.
    """
    # Dimension of the full Hilbert space
    full_dim = 2**L
    
    # Initialize the subsystem unitary
    haar_matrix = unitary_group.rvs(4)

    # Initialize the full unitary as an identity matrix
    full_unitary = np.eye(full_dim, dtype=complex)
    
    # Iterate over all possible states of the remaining qubits
    for remaining_state in range(2**(L-2)):
        # Binary representation of the remaining qubits
        remaining_bits = f"{remaining_state:0{L-2}b}"
        
        # Iterate over all possible 2-qubit states
        for i in range(4):
            for j in range(4):
                # Binary representation of the smaller indices
                binary_i = f"{i:02b}"
                binary_j = f"{j:02b}"
                
                # Initialize full state indices
                full_index_i = list(remaining_bits)
                full_index_j = list(remaining_bits)
                
                # Map the 2-qubit binary strings to the target qubits in the full state
                full_index_i.insert(target_qubits[0], binary_i[0])
                full_index_i.insert(target_qubits[1], binary_i[1])
                full_index_j.insert(target_qubits[0], binary_j[0])
                full_index_j.insert(target_qubits[1], binary_j[1])
                
                # Convert the full indices back to integer form
                full_index_i = int("".join(full_index_i), 2)
                full_index_j = int("".join(full_index_j), 2)
                
                # Assign the corresponding value from the Haar matrix to the full unitary
                full_unitary[full_index_i, full_index_j] = haar_matrix[i, j]
    
    return full_unitary

def simulate_bernoulli_map(x0, L, time):
    """
    Simulate the quantum Bernoulli map by applying cyclic shifts and unitary operations.
    
    Parameters:
    x0 (float): Initial real number to be digitized.
    L (int): Number of qubits in the system.
    time (int): Total time steps for the simulation.
    
    Returns:
    list: List of states after each time step.
    """
    # Digitize the initial state
    state = digitize_real_number(x0, L)
    
    # Convert to a full state vector in the computational basis
    full_state = np.zeros(2**L, dtype=complex)
    full_state[int("".join(map(str, state.astype(int))), 2)] = 1.0
    
    # Store the trajectory of the state
    trajectory = [full_state]
    
    # Perform the simulation
    for t in range(time):
        # Apply the cyclic leftward shift
        full_state = apply_cyclic_leftward_shift_to_state(full_state, L)
        
        # Get the last two qubit indices after the shift
        target_qubits = get_last_two_qubit_indices(t, L)
        
        # Apply the Haar random unitary to the last two qubits
        unitary = embed_haar_to_full_unitary(L, target_qubits)
        full_state = np.dot(unitary, full_state)
        
        # Normalize the state vector
        full_state /= np.linalg.norm(full_state)
        
        # Store the new state in the trajectory
        trajectory.append(full_state)
    
    return trajectory

# Parameters
x0 = 1 / np.sqrt(2)  # Initial real number
L = 4  # Number of qubits
time = 10  # Total time steps for the simulation

# Simulate the Bernoulli map
trajectory = simulate_bernoulli_map(x0, L, time)

# Print the final state
print("Final state vector after simulation:")
print(trajectory[-1])

Final state vector after simulation:
[-0.57763874-0.09744213j  0.31368014-0.24315445j  0.17962543+0.05189462j
 -0.10830509+0.06570177j -0.27757308+0.08534423j  0.08486437-0.17750417j
  0.18942522-0.05380497j -0.06012547+0.11909847j  0.22333193+0.24031217j
 -0.08166204+0.26394325j -0.06208291-0.08431985j  0.03548858-0.08072724j
  0.15240011+0.05677235j  0.02254488+0.135095j   -0.10248958-0.04071405j
 -0.01331126-0.09191602j]


In [127]:
import numpy as np

def apply_pauli_x_rightmost_qubit(L):
    """
    Constructs the Pauli-X gate that acts on the rightmost qubit in a system of L qubits.
    
    Parameters:
    L (int): The number of qubits.
    
    Returns:
    np.ndarray: The (2^L x 2^L) Pauli-X matrix acting on the rightmost qubit.
    """
    X_L = np.zeros((2**L, 2**L), dtype=complex)
    
    # Iterate over the first half of the basis states
    for i in range(2**L):
        if i % 2 == 0:
            X_L[i, i+1] = 1.0
            X_L[i+1, i] = 1.0
    
    return X_L

def measure_reset_rightmost_qubit(state, L):
    """
    Measure the rightmost qubit and reset it to the state |0>.
    
    Parameters:
    state (np.ndarray): The state vector in the computational basis.
    L (int): The number of qubits.
    
    Returns:
    np.ndarray: The state vector after the reset operation.
    """
    # Initialize the projector for the measurement P_L^0 and P_L^1
    P_L_0 = np.zeros((2**L, 2**L), dtype=complex)
    P_L_1 = np.zeros((2**L, 2**L), dtype=complex)
    
    # Define projectors for the rightmost qubit being 0 and 1
    for i in range(2**L):
        if i % 2 == 0:
            P_L_0[i, i] = 1.0
        else:
            P_L_1[i, i] = 1.0
    
    # Measure the rightmost qubit
    prob_0 = np.linalg.norm(np.dot(P_L_0, state))**2
    prob_1 = np.linalg.norm(np.dot(P_L_1, state))**2
    
    # Choose the outcome probabilistically based on the state's probabilities
    if np.random.rand() < prob_0:
        state = np.dot(P_L_0, state) / np.sqrt(prob_0)
    else:
        state = np.dot(P_L_1, state) / np.sqrt(prob_1)
        # Apply X gate if the measurement outcome was 1
        X_L = apply_pauli_x_rightmost_qubit(L)
        state = np.dot(X_L, state)
    
    return state

def apply_right_shift(state, L):
    """
    Applies the right shift operation T^{-1} to the state.
    
    Parameters:
    state (np.ndarray): The state vector in the computational basis.
    L (int): The number of qubits.
    
    Returns:
    np.ndarray: The new state vector after applying the right shift.
    """
    # Initialize the new state as a zero vector
    new_state = np.zeros_like(state, dtype=complex)
    
    # Iterate over each computational basis state
    for i in range(2**L):
        # Convert index to binary representation (bit string)
        bit_string = f"{i:0{L}b}"
        
        # Apply the right shift to the bit string
        shifted_bit_string = bit_string[-1] + bit_string[:-1]
        
        # Convert the shifted bit string back to an integer index
        shifted_index = int(shifted_bit_string, 2)
        
        # Update the new state vector
        new_state[shifted_index] = state[i]
    
    return new_state

def apply_control_map(state, L):
    """
    Apply the control map C to the quantum state.
    
    Parameters:
    state (np.ndarray): The state vector in the computational basis.
    L (int): The number of qubits.
    
    Returns:
    np.ndarray: The new state vector after applying the control map.
    """
    # Step 1: Reset the rightmost qubit to 0
    state = measure_reset_rightmost_qubit(state, L)
    
    # Step 2: Apply the right shift operation
    state = apply_right_shift(state, L)
    
    return state

# Example usage
L = 4  # Number of qubits

# Initialize a state, for example, |1101>
state = np.zeros(2**L, dtype=complex)
state[int("1101", 2)] = 1.0

# Apply the control map
new_state = apply_control_map(state, L)

# Print the new state
print("New state vector after applying the control map:")
print(new_state)

New state vector after applying the control map:
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
